<a href="https://colab.research.google.com/github/optiblackcode/amplitude_user_data_upload/blob/main/Burfy_data_ingestion_amplitude.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Cell 1 — Mount Drive & Imports

In [ ]:
import pandas as pd
import re
from google.colab import drive

drive.mount('/content/drive')
print('✅ Drive mounted')

Mounted at /content/drive
✅ Drive mounted


Cell 2 — Load file from Drive

In [ ]:
file_path = '/content/drive/MyDrive/burfy_amplitude/subscriptions burfy - cleaned data.csv'

df = pd.read_csv(file_path)

print(f'✅ Loaded: subscriptions burfy - cleaned data.csv')
print(f'   Rows    : {len(df):,}')
print(f'   Columns : {len(df.columns)}')
print(f'\n📋 Original headers:')
for col in df.columns:
    print(f'   • {col}')

✅ Loaded: subscriptions burfy - cleaned data.csv
   Rows    : 29,884
   Columns : 9

📋 Original headers:
   • user_id
   • os
   • plan_name
   • plan_id
   • subscription_id
   • paid_count
   • is_trial_availed
   • auto_renew_product_id
   • auto_renew_status


Cell 3 — snake_case conversion

In [ ]:
def to_snake_case(name: str) -> str:
    name = name.strip()
    name = re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', name)   # camelCase → camel_case
    name = re.sub(r'[\s\-\./()+]+', '_', name)              # spaces/hyphens → _
    name = re.sub(r'[^\w]', '', name)                       # remove special chars
    name = re.sub(r'_+', '_', name)                         # collapse __ → _
    return name.strip('_').lower()

rename_map = {col: to_snake_case(col) for col in df.columns}

print('🔄 Rename mapping:')
print(f'  {"ORIGINAL":<35} → SNAKE_CASE')
print(f'  {"-"*35}   {"-"*35}')
for original, snake in rename_map.items():
    changed = '✏️ ' if original != snake else '  '
    print(f'  {changed}{original:<33} → {snake}')

🔄 Rename mapping:
  ORIGINAL                            → SNAKE_CASE
  -----------------------------------   -----------------------------------
    user_id                           → user_id
    os                                → os
    plan_name                         → plan_name
    plan_id                           → plan_id
    subscription_id                   → subscription_id
    paid_count                        → paid_count
    is_trial_availed                  → is_trial_availed
    auto_renew_product_id             → auto_renew_product_id
    auto_renew_status                 → auto_renew_status


Cell 4 — Apply & Preview

In [ ]:
df_clean = df.rename(columns=rename_map)

print('✅ Headers renamed!')
print('\n📋 New headers:')
for col in df_clean.columns:
    print(f'   • {col}')

print('\n👀 First 3 rows preview:')
df_clean.head(3)

✅ Headers renamed!

📋 New headers:
   • user_id
   • os
   • plan_name
   • plan_id
   • subscription_id
   • paid_count
   • is_trial_availed
   • auto_renew_product_id
   • auto_renew_status

👀 First 3 rows preview:


,user_id,os,plan_name,plan_id,subscription_id,paid_count,is_trial_availed,auto_renew_product_id,auto_renew_status
0,03b2mzp4abPiaVfLzKS8OWWjgwq1,IOS,com.epicroots.burfyapp.weekly,plan_QJ3pPJfRMdySWX,200002917533565,49.0,False,com.epicroots.burfyapp.weekly,False
1,005ZTCiTkRhMKsisdRVDmT6TLZd2,Android,monthly,plan_Pw0Ztsj0xmr5Mb,order_RtlqcKxciD5tBX,NaN,NaN,NaN,NaN
2,009zTjbUIgNUvz2RbuyzvlUIMMt2,Android,monthly,plan_Pw0Ztsj0xmr5Mb,order_S0BAaP2ZshpM5t,NaN,NaN,NaN,NaN


Cell 5 — Save back to same Drive folder

In [ ]:
output_path = '/content/drive/MyDrive/burfy_amplitude/subscriptions_burfy_snake_case.csv'

df_clean.to_csv(output_path, index=False)
print(f'✅ Saved to Drive: {output_path}')

✅ Saved to Drive: /content/drive/MyDrive/burfy_amplitude/subscriptions_burfy_snake_case.csv


Cell 2 — Config

In [ ]:
API_KEY       = "88433fc3914de046e0aaa96292fcc536"
IDENTIFY_URL  = "https://api2.amplitude.com/identify"
BATCH_SIZE    = 100
RETRY_LIMIT   = 3
RETRY_DELAY   = 2
REQUEST_DELAY = 0.2

# ✅ Reading the already snake_case file
FILE_PATH = '/content/drive/MyDrive/burfy_amplitude/subscriptions_burfy_snake_case.csv'

print('✅ Config ready')

✅ Config ready


Cell 3 — Load & Validate

In [ ]:
df = pd.read_csv(FILE_PATH)

print(f'✅ Loaded: {len(df):,} rows, {len(df.columns)} columns')
print(f'\n📋 Headers:')
for col in df.columns:
    print(f'   • {col}')

print(f'\n👀 Preview:')
df.head(3)

✅ Loaded: 29,884 rows, 9 columns

📋 Headers:
   • user_id
   • os
   • plan_name
   • plan_id
   • subscription_id
   • paid_count
   • is_trial_availed
   • auto_renew_product_id
   • auto_renew_status

👀 Preview:


,user_id,os,plan_name,plan_id,subscription_id,paid_count,is_trial_availed,auto_renew_product_id,auto_renew_status
0,03b2mzp4abPiaVfLzKS8OWWjgwq1,IOS,com.epicroots.burfyapp.weekly,plan_QJ3pPJfRMdySWX,200002917533565,49.0,False,com.epicroots.burfyapp.weekly,False
1,005ZTCiTkRhMKsisdRVDmT6TLZd2,Android,monthly,plan_Pw0Ztsj0xmr5Mb,order_RtlqcKxciD5tBX,NaN,NaN,NaN,NaN
2,009zTjbUIgNUvz2RbuyzvlUIMMt2,Android,monthly,plan_Pw0Ztsj0xmr5Mb,order_S0BAaP2ZshpM5t,NaN,NaN,NaN,NaN


Cell 4:

In [ ]:
# ── os: normalize casing ──────────────────────────
df['os'] = df['os'].str.strip().str.upper()   # IOS, ANDROID consistently

# ── paid_count: float → int (keep NaN rows as missing, don't send 0) ──
df['paid_count'] = pd.to_numeric(df['paid_count'], errors='coerce')
# Leave as float — we'll drop NaN fields per-row during payload build

# ── str cols: strip whitespace ────────────────────
STR_COLS = ['user_id', 'plan_name', 'plan_id', 'subscription_id', 'auto_renew_product_id']
for col in STR_COLS:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().replace('nan', pd.NA)

# ── Drop rows with missing user_id ────────────────
before = len(df)
df = df[df['user_id'].notna() & (df['user_id'] != '')]
skipped = before - len(df)

print(f'✅ Coercions done')
print(f'   Valid rows  : {len(df):,}')
print(f'   Skipped     : {skipped} (missing user_id)')
print(f'\n📊 os distribution:')
print(df['os'].value_counts(dropna=False).to_string())
print(f'\n📊 paid_count nulls : {df["paid_count"].isna().sum():,}')
print(f'📊 is_trial_availed nulls : {df["is_trial_availed"].isna().sum():,}')
print(f'📊 auto_renew_status nulls: {df["auto_renew_status"].isna().sum():,}')

✅ Coercions done
   Valid rows  : 29,547
   Skipped     : 337 (missing user_id)

📊 os distribution:
os
ANDROID    25934
IOS         3058
NaN          555

📊 paid_count nulls : 13,754
📊 is_trial_availed nulls : 25,215
📊 auto_renew_status nulls: 26,432


cell 5 : payload builder

In [ ]:
import json

PREVIEW_ROWS = 3
sample_batch = []

for _, row in df.head(PREVIEW_ROWS).iterrows():
    props = {}
    for k, v in row.to_dict().items():
        # Skip nulls
        if not isinstance(v, bool) and pd.isna(v):
            continue
        if isinstance(v, str) and v.strip() in ('', 'nan'):
            continue
        # paid_count float → int
        if k == 'paid_count':
            props[k] = int(v)
        else:
            props[k] = v

    sample_batch.append({
        "user_id":         props["user_id"],
        "user_properties": props
    })

print('📦 Sample Amplitude Identify Payload')
print('='*60)
print(json.dumps(sample_batch, indent=2, default=str))
print('='*60)
print(f'\n   Total valid rows  : {len(df):,}')
print(f'   Batch size        : 100')
print(f'   Total batches     : {-(-len(df) // 100):,}')
print('\n⚠️  Review payload above — confirm before running Cell 6')

📦 Sample Amplitude Identify Payload
[
  {
    "user_id": "03b2mzp4abPiaVfLzKS8OWWjgwq1",
    "user_properties": {
      "user_id": "03b2mzp4abPiaVfLzKS8OWWjgwq1",
      "os": "IOS",
      "plan_name": "com.epicroots.burfyapp.weekly",
      "plan_id": "plan_QJ3pPJfRMdySWX",
      "subscription_id": "200002917533565",
      "paid_count": 49,
      "is_trial_availed": false,
      "auto_renew_product_id": "com.epicroots.burfyapp.weekly",
      "auto_renew_status": false
    }
  },
  {
    "user_id": "005ZTCiTkRhMKsisdRVDmT6TLZd2",
    "user_properties": {
      "user_id": "005ZTCiTkRhMKsisdRVDmT6TLZd2",
      "os": "ANDROID",
      "plan_name": "monthly",
      "plan_id": "plan_Pw0Ztsj0xmr5Mb",
      "subscription_id": "order_RtlqcKxciD5tBX"
    }
  },
  {
    "user_id": "009zTjbUIgNUvz2RbuyzvlUIMMt2",
    "user_properties": {
      "user_id": "009zTjbUIgNUvz2RbuyzvlUIMMt2",
      "os": "ANDROID",
      "plan_name": "monthly",
      "plan_id": "plan_Pw0Ztsj0xmr5Mb",
      "subscription_id

Cell 6:

In [ ]:
import requests
import json
import time
import pandas as pd

def send_batch(batch, attempt=1):
    payload = {
        "api_key":        API_KEY,
        "identification": json.dumps(batch, default=str)
    }
    try:
        resp = requests.post(IDENTIFY_URL, data=payload, timeout=15)
        # ✅ Amplitude returns either "1" or "success"
        if resp.status_code == 200 and resp.text.strip() in ("1", "success"):
            return True
        print(f'  ⚠️  Attempt {attempt}: HTTP {resp.status_code} — {resp.text[:200]}')
        if attempt < RETRY_LIMIT:
            time.sleep(RETRY_DELAY * attempt)
            return send_batch(batch, attempt + 1)
        return False
    except requests.RequestException as e:
        print(f'  ❌ Network error: {e}')
        if attempt < RETRY_LIMIT:
            time.sleep(RETRY_DELAY * attempt)
            return send_batch(batch, attempt + 1)
        return False

# ── Ingestion loop ─────────────────────────────────
total      = len(df)
success    = 0
failed     = 0
failed_ids = []
batch      = []
batch_num  = 0

for _, row in df.iterrows():
    props = {}
    for k, v in row.to_dict().items():
        if not isinstance(v, bool) and pd.isna(v):
            continue
        if isinstance(v, str) and v.strip() in ('', 'nan'):
            continue
        if k == 'paid_count':
            props[k] = int(v)
        else:
            props[k] = v

    batch.append({
        "user_id":         props["user_id"],
        "user_properties": props
    })

    if len(batch) >= BATCH_SIZE:
        batch_num += 1
        ok = send_batch(batch)
        if ok:
            success += len(batch)
            print(f'✅ Batch {batch_num:>3} — {success:,}/{total:,} sent')
        else:
            failed += len(batch)
            failed_ids += [b["user_id"] for b in batch]
            print(f'❌ Batch {batch_num:>3} FAILED')
        batch = []
        time.sleep(REQUEST_DELAY)

# Remaining rows
if batch:
    batch_num += 1
    ok = send_batch(batch)
    if ok:
        success += len(batch)
        print(f'✅ Batch {batch_num:>3} — {success:,}/{total:,} sent')
    else:
        failed += len(batch)
        failed_ids += [b["user_id"] for b in batch]
        print(f'❌ Final batch FAILED')

print(f'\n{"="*50}')
print(f'✅ DONE')
print(f'   Total   : {total:,}')
print(f'   Success : {success:,}')
print(f'   Failed  : {failed:,}')
print(f'{"="*50}')

✅ Batch   1 — 100/29,547 sent
✅ Batch   2 — 200/29,547 sent
✅ Batch   3 — 300/29,547 sent
✅ Batch   4 — 400/29,547 sent
✅ Batch   5 — 500/29,547 sent
✅ Batch   6 — 600/29,547 sent
✅ Batch   7 — 700/29,547 sent
✅ Batch   8 — 800/29,547 sent
✅ Batch   9 — 900/29,547 sent
✅ Batch  10 — 1,000/29,547 sent
✅ Batch  11 — 1,100/29,547 sent
✅ Batch  12 — 1,200/29,547 sent
✅ Batch  13 — 1,300/29,547 sent
✅ Batch  14 — 1,400/29,547 sent
✅ Batch  15 — 1,500/29,547 sent
✅ Batch  16 — 1,600/29,547 sent
✅ Batch  17 — 1,700/29,547 sent
✅ Batch  18 — 1,800/29,547 sent
✅ Batch  19 — 1,900/29,547 sent
✅ Batch  20 — 2,000/29,547 sent
✅ Batch  21 — 2,100/29,547 sent
✅ Batch  22 — 2,200/29,547 sent
✅ Batch  23 — 2,300/29,547 sent
✅ Batch  24 — 2,400/29,547 sent
✅ Batch  25 — 2,500/29,547 sent
✅ Batch  26 — 2,600/29,547 sent
✅ Batch  27 — 2,700/29,547 sent
✅ Batch  28 — 2,800/29,547 sent
✅ Batch  29 — 2,900/29,547 sent
✅ Batch  30 — 3,000/29,547 sent
✅ Batch  31 — 3,100/29,547 sent
✅ Batch  32 — 3,200/29,547

Inspecting file for 337 ids

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

df_missing = pd.read_csv('/content/drive/MyDrive/burfy_amplitude/subscriptions burfy - 337_records.csv')

print(f'Rows: {len(df_missing)}')
print(f'\n📋 Columns: {df_missing.columns.tolist()}')
print(f'\n📊 Null counts:')
print(df_missing.isnull().sum().to_string())
print(f'\n👀 Sample rows:')
df_missing.head(5)

Rows: 337

📋 Columns: ['user_id', 'os', 'plan_name', 'plan_id', 'subscription_id', 'paid_count', 'is_trial_availed', 'auto_renew_product_id', 'auto_renew_status']

📊 Null counts:
user_id                    0
os                        15
plan_name                  0
plan_id                  336
subscription_id            1
paid_count                14
is_trial_availed         331
auto_renew_product_id      1
auto_renew_status          1

👀 Sample rows:


,user_id,os,plan_name,plan_id,subscription_id,paid_count,is_trial_availed,auto_renew_product_id,auto_renew_status
0,08EHE5P8fFgpn2tm8U3TiY0Pv9N2,IOS,com.epicroots.burfyapp.monthly,NaN,7.000303e+13,210.0,NaN,com.epicroots.burfyapp.monthly,True
1,0EL4HjUTyhOqHxshjxqbcIBZvug2,IOS,com.epicroots.burfyapp.monthly,NaN,5.100023e+14,9.0,NaN,com.epicroots.burfyapp.monthly,True
2,0GquE4R3IphRL8pH5j92LGYhJ7I2,IOS,com.epicroots.burfyapp.monthly,NaN,4.100028e+14,8.0,NaN,com.epicroots.burfyapp.monthly,False
3,0Hoc8EzPN0hKMgY7r7b5yxuIJ4r2,IOS,com.epicroots.burfyapp.weekly,NaN,4.600027e+14,39.0,NaN,com.epicroots.burfyapp.weekly,True
4,0MuUJIfAysPbQ2ICNlaUcFVyjpF3,IOS,com.epicroots.burfyapp.weekly,NaN,5.200024e+14,5.0,NaN,com.epicroots.burfyapp.weekly,True


In [ ]:
import pandas as pd
import requests
import json
import time

# ── Config ─────────────────────────────────────────
API_KEY       = "88433fc3914de046e0aaa96292fcc536"
IDENTIFY_URL  = "https://api2.amplitude.com/identify"
BATCH_SIZE    = 100
RETRY_LIMIT   = 3
RETRY_DELAY   = 2
REQUEST_DELAY = 0.2

# ── Load with subscription_id forced as string ─────
df_missing = pd.read_csv(
    '/content/drive/MyDrive/burfy_amplitude/subscriptions burfy - 337_records.csv',
    dtype={'subscription_id': str}   # ← prevents scientific notation
)

# ── Coercions ──────────────────────────────────────
df_missing['os'] = df_missing['os'].str.strip().str.upper()

for col in ['is_trial_availed', 'auto_renew_status']:
    if col in df_missing.columns:
        df_missing[col] = df_missing[col].astype(str).str.strip().str.lower().isin(['true', '1', 'yes'])

df_missing['paid_count'] = pd.to_numeric(df_missing['paid_count'], errors='coerce')

for col in ['user_id', 'plan_name', 'plan_id', 'auto_renew_product_id']:
    if col in df_missing.columns:
        df_missing[col] = df_missing[col].astype(str).str.strip().replace('nan', pd.NA)

print(f'✅ Loaded {len(df_missing)} rows')
print(f'\n👀 subscription_id sample (should be string now):')
print(df_missing['subscription_id'].head(5).tolist())

✅ Loaded 337 rows

👀 subscription_id sample (should be string now):
['70003032530188', '510002279758490', '410002768994810', '460002741530689', '520002420466042']


send cell

In [ ]:
def send_batch(batch, attempt=1):
    payload = {
        "api_key":        API_KEY,
        "identification": json.dumps(batch, default=str)
    }
    try:
        resp = requests.post(IDENTIFY_URL, data=payload, timeout=15)
        if resp.status_code == 200 and resp.text.strip() in ("1", "success"):
            return True
        print(f'  ⚠️  Attempt {attempt}: HTTP {resp.status_code} — {resp.text[:200]}')
        if attempt < RETRY_LIMIT:
            time.sleep(RETRY_DELAY * attempt)
            return send_batch(batch, attempt + 1)
        return False
    except requests.RequestException as e:
        print(f'  ❌ Network error: {e}')
        if attempt < RETRY_LIMIT:
            time.sleep(RETRY_DELAY * attempt)
            return send_batch(batch, attempt + 1)
        return False

# ── Ingestion loop ─────────────────────────────────
total      = len(df_missing)
success    = 0
failed     = 0
failed_ids = []
batch      = []
batch_num  = 0

for _, row in df_missing.iterrows():
    props = {}
    for k, v in row.to_dict().items():
        if not isinstance(v, bool) and pd.isna(v):
            continue
        if isinstance(v, str) and v.strip() in ('', 'nan'):
            continue
        if k == 'paid_count':
            props[k] = int(v)
        else:
            props[k] = v

    batch.append({
        "user_id":         props["user_id"],
        "user_properties": props
    })

    if len(batch) >= BATCH_SIZE:
        batch_num += 1
        ok = send_batch(batch)
        if ok:
            success += len(batch)
            print(f'✅ Batch {batch_num} — {success}/{total} sent')
        else:
            failed += len(batch)
            failed_ids += [b["user_id"] for b in batch]
            print(f'❌ Batch {batch_num} FAILED')
        batch = []
        time.sleep(REQUEST_DELAY)

# Remaining
if batch:
    batch_num += 1
    ok = send_batch(batch)
    if ok:
        success += len(batch)
        print(f'✅ Final batch — {success}/{total} sent')
    else:
        failed += len(batch)
        failed_ids += [b["user_id"] for b in batch]

print(f'\n{"="*50}')
print(f'✅ DONE')
print(f'   Total   : {total}')
print(f'   Success : {success}')
print(f'   Failed  : {failed}')
print(f'{"="*50}')

✅ Batch 1 — 100/337 sent
✅ Batch 2 — 200/337 sent
✅ Batch 3 — 300/337 sent
✅ Final batch — 337/337 sent

✅ DONE
   Total   : 337
   Success : 337
   Failed  : 0
